<a href="https://colab.research.google.com/github/syedsaliq7866/basketball-cv-tracking/blob/main/basketball_cv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
!pip install -q ultralytics opencv-python supervision pandas

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==========================================
# 1. Front-Left (FL) 5-Minute Tracking Loop
# ==========================================
import os
import cv2
import pandas as pd
from ultralytics import YOLO

drive_out_dir = '/content/drive/MyDrive/basketball_cv/outputs_full/'
os.makedirs(drive_out_dir, exist_ok=True)

video_path = '/content/drive/MyDrive/basketball_cv/10-2 FL_5min.mp4'
output_video_path = os.path.join(drive_out_dir, 'FL_full_5min_tracked_v2.mp4')
csv_path = os.path.join(drive_out_dir, 'FL_full_5min_tracks_v2.csv')

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

model = YOLO('yolov8s.pt')
records = []
frame_idx = 0

print("Starting FL Tracking with sideline filters & confidence labels...")

for result in model.track(
    source=video_path,
    classes=[0, 32],
    tracker="bytetrack.yaml",
    conf=0.05,
    imgsz=960,
    persist=True,
    stream=True,
    verbose=False
):
    frame = result.orig_img.copy()

    if result.boxes is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.int().cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()
        track_ids = result.boxes.id.int().cpu().numpy() if result.boxes.id is not None else [None] * len(boxes)

        for box, cls_id, conf, tid in zip(boxes, classes, confs, track_ids):
            x1, y1, x2, y2 = map(int, box)

            # Spatial Filters for FL: Exclude Kobe mural and sideline bench
            if x1 > int(width * 0.82) and y1 < int(height * 0.60):
                continue
            if x2 < int(width * 0.18):
                continue

            if cls_id == 0 and conf >= 0.45:
                id_str = f"id:{tid} " if tid is not None else ""
                label = f"{id_str}person {conf:.2f}"
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, max(20, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0, 255, 0), 2)
                records.append({
                    'frame': frame_idx, 'timestamp': round(frame_idx / fps, 2),
                    'camera': 'FL', 'class': 'player', 'local_id': int(tid) if tid is not None else -1,
                    'confidence': round(float(conf), 2), 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2
                })

            elif cls_id == 32 and conf >= 0.05:
                bw, bh = x2 - x1, y2 - y1
                if bw > 120 or bh > 120:
                    continue
                label = f"ball {conf:.2f}"
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 140, 255), 2)
                cv2.putText(frame, label, (x1, max(20, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0, 140, 255), 2)
                records.append({
                    'frame': frame_idx, 'timestamp': round(frame_idx / fps, 2),
                    'camera': 'FL', 'class': 'ball', 'local_id': -1,
                    'confidence': round(float(conf), 2), 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2
                })

    out.write(frame)
    frame_idx += 1
    if frame_idx % 500 == 0 or frame_idx == total_frames:
        pct = (frame_idx / total_frames) * 100
        print(f"FL Progress: {frame_idx}/{total_frames} frames ({pct:.1f}%)")

cap.release()
out.release()
pd.DataFrame(records).to_csv(csv_path, index=False)
print("FL 5-Minute Processing Complete!")

In [ ]:
# ==========================================
# 2. Near-Left (NL) 5-Minute Tracking Loop
# ==========================================
import os
import cv2
import pandas as pd
from ultralytics import YOLO

drive_out_dir = '/content/drive/MyDrive/basketball_cv/outputs_full/'
os.makedirs(drive_out_dir, exist_ok=True)

video_path = '/content/drive/MyDrive/basketball_cv/10-2 NL_5min.mp4'
output_video_path = os.path.join(drive_out_dir, 'NL_full_5min_tracked.mp4')
csv_path = os.path.join(drive_out_dir, 'NL_full_5min_tracks.csv')

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

model = YOLO('yolov8s.pt')
records = []
frame_idx = 0

print("Starting NL Processing...")

for result in model.track(
    source=video_path,
    classes=[0, 32],
    tracker="bytetrack.yaml",
    conf=0.05,
    imgsz=960,
    persist=True,
    stream=True,
    verbose=False
):
    frame = result.orig_img.copy()

    if result.boxes is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.int().cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()
        track_ids = result.boxes.id.int().cpu().numpy() if result.boxes.id is not None else [None] * len(boxes)

        for box, cls_id, conf, tid in zip(boxes, classes, confs, track_ids):
            x1, y1, x2, y2 = map(int, box)

            if cls_id == 0 and conf >= 0.45:
                id_str = f"id:{tid} " if tid is not None else ""
                label = f"{id_str}person {conf:.2f}"
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, label, (x1, max(20, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0, 255, 0), 2)
                records.append({
                    'frame': frame_idx, 'timestamp': round(frame_idx / fps, 2),
                    'camera': 'NL', 'class': 'player', 'local_id': int(tid) if tid is not None else -1,
                    'confidence': round(float(conf), 2), 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2
                })

            elif cls_id == 32 and conf >= 0.05:
                bw, bh = x2 - x1, y2 - y1
                if bw > 120 or bh > 120:
                    continue
                label = f"ball {conf:.2f}"
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 140, 255), 2)
                cv2.putText(frame, label, (x1, max(20, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0, 140, 255), 2)
                records.append({
                    'frame': frame_idx, 'timestamp': round(frame_idx / fps, 2),
                    'camera': 'NL', 'class': 'ball', 'local_id': -1,
                    'confidence': round(float(conf), 2), 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2
                })

    out.write(frame)
    frame_idx += 1
    if frame_idx % 500 == 0 or frame_idx == total_frames:
        pct = (frame_idx / total_frames) * 100
        print(f"NL Progress: {frame_idx}/{total_frames} frames ({pct:.1f}%)")

cap.release()
out.release()
pd.DataFrame(records).to_csv(csv_path, index=False)
print("NL Processing Complete! Ready for Cross-Camera ID Matching.")